# Leakage Review for `baseline_bank_marketing_v2.py`

This notebook documents the methodological issues in `baseline_bank_marketing_v2.py`, explains why they are invalid, and runs a corrected evaluation that preserves the original workflow as much as possible.

## Detected issues

1. `job_yes_rate` is computed using the full dataset, including test labels.
2. The preprocessor is fit on the full feature matrix before the train/test split.
3. `duration` is included by default, even though it is typically unavailable at real scoring time for this task.

In [ ]:
from pathlib import Path

path = Path('baseline_bank_marketing_v2.py')
text = path.read_text()
print(text[:4000])

## Why these issues are invalid

- **Target leakage in `job_yes_rate`**: the feature is built using `y` from all rows, so test labels affect a predictor used during evaluation.
- **Pre-split preprocessing leakage**: imputation, scaling, and one-hot encoding are fit on the full dataset before splitting, so the training process sees information from the test set.
- **Deployment-time leakage via `duration`**: if the model is intended to score before or at the start of contact, `duration` is not available and should not be used.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, balanced_accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42


def infer_feature_types(df: pd.DataFrame, target: str) -> tuple[list[str], list[str]]:
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target]
    categorical_cols = [c for c in df.columns if c not in numeric_cols + [target]]
    return numeric_cols, categorical_cols


def replace_unknown_tokens(df: pd.DataFrame, categorical_cols: list[str]) -> pd.DataFrame:
    clean = df.copy()
    for col in categorical_cols:
        clean[col] = clean[col].replace('unknown', np.nan)
    return clean


def build_preprocessor(numeric_cols: list[str], categorical_cols: list[str]) -> ColumnTransformer:
    numeric_pipe = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ('num', numeric_pipe, numeric_cols),
            ('cat', categorical_pipe, categorical_cols),
        ]
    )


def metric_row(model_name: str, y_true: pd.Series, y_pred: np.ndarray, y_score: np.ndarray) -> dict:
    return {
        'model': model_name,
        'roc_auc': roc_auc_score(y_true, y_score),
        'pr_auc': average_precision_score(y_true, y_score),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'positive_rate_pred': float(np.mean(y_pred)),
    }


## Correct evaluation procedure

The corrected workflow is:

1. Load the raw data.
2. Drop `duration`.
3. Replace `unknown` with missing values.
4. Split raw `X` and `y` into train/test.
5. Compute `job_yes_rate` on the training split only, then map it into train and test.
6. Fit the preprocessor on `X_train` only, then transform `X_train` and `X_test`.
7. Fit the models on training data only and evaluate on the untouched test split.

In [ ]:
df = pd.read_csv('bank-additional-full.csv', sep=';')
working_df = df.drop(columns=['duration']).copy()

base_categorical_cols = [c for c in working_df.columns if working_df[c].dtype == 'object' and c != 'y']
working_df = replace_unknown_tokens(working_df, base_categorical_cols)

y = working_df['y'].map({'no': 0, 'yes': 1})
X = working_df.drop(columns=['y']).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

train_job_yes_rate = X_train.assign(y=y_train).groupby('job')['y'].mean()
global_rate = y_train.mean()

X_train['job_yes_rate'] = X_train['job'].map(train_job_yes_rate).fillna(global_rate)
X_test['job_yes_rate'] = X_test['job'].map(train_job_yes_rate).fillna(global_rate)

train_df_for_types = pd.concat([X_train, y_train.rename('y')], axis=1)
numeric_cols, categorical_cols = infer_feature_types(train_df_for_types, 'y')

preprocessor = build_preprocessor(numeric_cols, categorical_cols)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

dummy = DummyClassifier(strategy='prior')
logistic = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=SEED,
    solver='lbfgs',
)

dummy.fit(X_train_processed, y_train)
logistic.fit(X_train_processed, y_train)

dummy_pred = dummy.predict(X_test_processed)
dummy_score = dummy.predict_proba(X_test_processed)[:, 1]
log_pred = logistic.predict(X_test_processed)
log_score = logistic.predict_proba(X_test_processed)[:, 1]

results = pd.DataFrame([
    metric_row('DummyClassifier(prior)', y_test, dummy_pred, dummy_score),
    metric_row('LogisticRegression(balanced)', y_test, log_pred, log_score),
]).sort_values('pr_auc', ascending=False)

results

In [ ]:
report_df = pd.DataFrame(
    classification_report(y_test, log_pred, target_names=['no', 'yes'], output_dict=True)
).T.reset_index(names='label')
report_df

## How corrected results may differ

The corrected results should usually be lower than the original `baseline_bank_marketing_v2.py` outputs because:

- the model no longer sees target-derived information from the test set through `job_yes_rate`,
- the preprocessor no longer learns imputation/scaling/encoding structure from the test set,
- and `duration` is removed, which avoids a post-contact leakage variable.

That drop is expected and indicates a more credible estimate of real-world performance.